# Example Figures (Paper)

In [ ]:
%load_ext autoreload
%autoreload 2
import os

import math
from matplotlib import pyplot as plt
from torch import Tensor

from privtrajgen import config as conf
from privtrajgen.config import Datasets
from privtrajgen.datasets import denormalise_by_dataset, dataset_bounding_boxes
from privtrajgen.evaluation.factories import get_model, generate_trajectories, load_parameters
from privtrajgen.evaluation.factories import get_test_dataset, get_test_context
from privtrajgen.models.naming import generate_model_name
from privtrajgen.plot.tables import *
from privtrajgen.utils import logger, printc
from privtrajgen.utils.nn_utils import get_device

In [ ]:
# Logging
log = logger.configure_root_loger(logging.INFO)


def logc(header: str, *args: Any, column_width: int = 60) -> None:
    log.info(f"{header:<{column_width}}" + "".join([str(arg) for arg in args]))

## Configuration

In [ ]:
class Mode(Enum):
    PAPER = "paper"
    THESIS = "thesis"


class Format(Enum):
    COLUMN = 1
    PAGE = 2
    THESIS = 3


def pt_to_cm(pt):
    return pt * 0.0352778


def cm_to_inch(cm):
    """Matplotlib figsize accepts inches"""
    return cm / 2.54


def pt_to_inch(pt):
    return cm_to_inch(pt_to_cm(pt))

In [ ]:
# Configuration
USE_CACHE = True
OUTPUT_FORMAT = "png"  # PDF doesn't work for such huge plots.
MODE = Mode.PAPER
FORMAT = Format.COLUMN
OUTPUT_DIR = os.path.join(conf.BASE_DIR, 'img/')
CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache/')
OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'paper/examples/')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
# Configure matplotlib
if MODE == Mode.PAPER:
    # ACM
    column_width_pt = 241.14749  # 241.14749pt
    page_width_pt = 506.295  # 506.295pt

    column_width = pt_to_inch(column_width_pt)
    page_width = pt_to_inch(page_width_pt)
    thesis_width = None

    fig_width = column_width if FORMAT == Format.COLUMN else page_width
    fig_height = cm_to_inch(2.5)

    FONT_SIZE = 18
    plt.rcParams.update({
        'font.size': FONT_SIZE,  # Global font size
        'axes.labelsize': FONT_SIZE,  # Increase axis labels font by 2
        'axes.titlesize': FONT_SIZE,  # Set title font size to same as axis labels
        'xtick.labelsize': FONT_SIZE - 3,  # Tick-label font size
        'ytick.labelsize': FONT_SIZE - 3,  # Tick-label font size
        'legend.fontsize': FONT_SIZE,  # Make the legend/label fonts a little smaller
        'figure.subplot.wspace': 0.05,  # Reduce space between subfigures
        'figure.subplot.hspace': 0.05,  # Reduce vertical space to 0.1
        'figure.figsize': (column_width, fig_height),  # Default figure size
    })

    # Values not in rcParams
    SCATTER_SIZE = 50

elif MODE == Mode.THESIS:
    # Thesis Dimensions (Note that matplotlib uses inches, use plot.cm to convert to cm)
    LINEWIDTH_THESIS = 15.0562504028  # in cm, equals 426.79135 pt
    HALF_LINEWIDTH_THESIS = 7.3776451944  # in cm, equals 213.395675 pt
    # Thesis font sizes
    # The thesis used font 'LM Roman 10' with 10.95pt for captions and 10pt for sub-captions
    # In the default matplotlib font this equals

    FONT_SIZE = 9
    # Update all relevant font size rcParams
    plt.rcParams.update({
        'font.size': FONT_SIZE,  # Global font size
        'axes.titlesize': FONT_SIZE,  # Title font size
        'axes.labelsize': FONT_SIZE,  # X and Y axis labels
        'xtick.labelsize': FONT_SIZE - 2,  # X-axis tick labels
        'ytick.labelsize': FONT_SIZE - 2,  # Y-axis tick labels
        'legend.fontsize': FONT_SIZE,  # Legend font size
        'figure.titlesize': FONT_SIZE,  # Figure title font size
        'figure.labelsize': FONT_SIZE,  # Figure label font size
        'lines.markersize': 3,  # Adjust marker size
        'figure.subplot.wspace': 0.1,  # Reduce space between subfigures
        'figure.subplot.hspace': 0.1,  # Reduce vertical space to 0.1
    })

    # Values not in rcParams
    SCATTER_SIZE = 30

    fig_width = cm_to_inch(LINEWIDTH_THESIS)
    aspect_ratio = 2.2
    fig_height = fig_width / aspect_ratio
else:
    raise ValueError("Invalid mode")

print(f"Width:\t\t\t{fig_width:.2f} inch\t=\t{fig_width * 2.54:.2f} cm")
print(f"Height:\t\t\t{fig_height:.2f} inch\t=\t{fig_height * 2.54:.2f} cm")
print(f"Aspect ratio:\t{fig_width / fig_height:.2f}")

page_width = 30

## Example Plots

In [ ]:
#########################################
# Constants
#########################################
N_TRAJ = 1000
GPU = 0


def load_cache(case: str | Datasets) -> Optional[np.ndarray]:
    if case in final_evaluation:
        # Not a dataset
        config = final_evaluation[case]
        name = generate_model_name(config)
    elif isinstance(case, Datasets):
        # Dataset
        name = case
    else:
        raise ValueError(f"Invalid case: {case}")

    cache_file = os.path.join(CACHE_DIR, f"{name}.npy")
    if os.path.exists(cache_file):
        # logc(f"Loaded cache for {case}: ", cache_file)
        log.info(f"Loaded cache for {case}.")
        return np.load(cache_file)
    else:
        return None


def save_cache(case: str | Datasets, trajs: np.ndarray) -> None:
    if case in final_evaluation:
        # Not a dataset
        config = final_evaluation[case]
        name = generate_model_name(config)
    elif isinstance(case, Datasets):
        # Dataset
        name = case
    else:
        raise ValueError(f"Invalid case: {case}")

    cache_file = os.path.join(CACHE_DIR, f"{name}.npy")
    np.save(cache_file, trajs)
    # logc(f"Saved cache for {case}: ", cache_file)
    log.info(f"Saved cache for {case}.")


def get_trajs(
        case: str | Datasets,
        guide: Optional[str] = None,
        n_traj: int = N_TRAJ,
        use_cache: bool = USE_CACHE,
        save_to_cache: bool = True,
) -> np.ndarray:
    if use_cache:
        cache = load_cache(case)
        if cache is not None:
            return cache

    if case in final_evaluation:
        # Don't use cache or no cache found
        config = final_evaluation[case]
        # Set GPU
        config.gpu = GPU
        # Set model name
        config.name = generate_model_name(config)
        # Retrieve the model
        model = get_model(config)
        # Move the model to the device
        model.to(get_device(config.gpu))
        # Load the model parameters
        model = load_parameters(config, model)
        # Load context
        head: Union[None, Tensor] = get_test_context(config)
        if head is not None:
            idx = np.random.choice(len(head), n_traj, replace=False)
            head = head[idx]
        # Generate trajectories
        model.eval()
        trajs = generate_trajectories(config, model, num=n_traj, head=head)
    elif isinstance(case, Datasets):
        assert guide is not None, "Guide must be provided for dataset"
        config = final_evaluation[guide]
        # Load dataset
        real_ds = get_test_dataset(config)
        # Select n_traj random trajectories
        idx = np.random.choice(len(real_ds), n_traj, replace=False)
        trajs = real_ds[idx]
    else:
        raise ValueError(f"Invalid case: {case}")

    # Reshape from (B, 2, L) to (B, L, 2)
    trajs = trajs.transpose(0, 2, 1)

    # Denormalise
    trajs = denormalise_by_dataset(trajs, config.dataset)

    # Save cache
    if save_to_cache:
        save_cache(case, trajs)

    return trajs


def plot_case(
        trajs: np.ndarray,
        subtitle: str,
        ax: plt.Axes,
):
    # Transform Trajectories to pointcloud
    trajs = trajs.reshape(-1, 2)
    # Plot
    ax.scatter(trajs[:, 0], trajs[:, 1], alpha=0.1, s=0.05)
    # Add title
    ax.set_title(subtitle)


def example_plots(
        cases: List[str],
        name: str,
        subtitles: List[str],
        n_rows: int = 1,
        use_cache: bool = USE_CACHE,
        save_to_cache: bool = True,
        title: Optional[str] = None,
        include_dataset: bool = True,
):
    """
    Generate sample trajectories for all given cases and plot them next to each other.
    The first plot contains the dataset.
    """
    # Determine dataset
    dataset = final_evaluation[cases[0]].dataset
    # Determine fold
    fold = final_evaluation[cases[0]].run_id
    n_kfold = final_evaluation[cases[0]].k_fold

    # Check that the subtitles match the number of cases
    assert len(subtitles) is None or len(subtitles) == len(cases), "Number of subtitles must match number of cases"
    # Check that all cases use the same dataset
    for case in cases:
        assert final_evaluation[case].dataset == dataset, f"Dataset mismatch for case {case}"
        assert final_evaluation[case].run_id == fold, f"Fold mismatch for case {case}"
        assert final_evaluation[case].k_fold == n_kfold, f"K-Fold mismatch for case {case}"

    # Add dataset as first plot
    if include_dataset:
        cases = [dataset] + cases
        subtitles = ['Dataset'] + subtitles

    # Create Plot
    n_col = math.ceil(len(cases) / n_rows)
    fig, axs = plt.subplots(n_rows, n_col, figsize=(page_width, page_width / n_col), sharey=True, sharex=True)

    # Plot the cases
    for i, (case, subtitle) in enumerate(zip(cases, subtitles)):
        # Load Data
        guide = cases[1] if isinstance(case, Datasets) else None
        try:
            trajs = get_trajs(case, guide, use_cache=use_cache, save_to_cache=save_to_cache)
        except FileNotFoundError as e:
            # Data missing - just display an empty subplot
            # This is deactivated for the final plots!
            # continue
            raise e
        # Plot
        ax = axs[i // n_col, i % n_col] if n_rows > 1 else axs[i]
        plot_case(trajs, subtitle, ax)

    # Set title
    if title is not None:
        fig.suptitle(title, y=0.95)

    # Set Ticks
    # Retrieve the dataset-specific bounding box
    bounding_box = dataset_bounding_boxes[dataset]
    # Synced so set for first ax only
    ax = axs[0, 0] if n_rows > 1 else axs[0]
    ax.set_xlim(bounding_box["lon_min"], bounding_box["lon_max"])
    ax.set_ylim(bounding_box["lat_min"], bounding_box["lat_max"])

    # Tight layout
    plt.tight_layout()

    # Save figure
    output_file: str = os.path.join(OUTPUT_DIR, f"{name}.{OUTPUT_FORMAT}")
    plt.savefig(os.path.join(output_file), dpi=300, bbox_inches="tight")
    logc(f"Saved plot: ", os.path.join(output_file))
    plt.show()

# # Call it with example data
# example = ['7p', '7p', '7p', '7p', '7p']
# example_plots(example, "Example", ["A", "B", "C", "D", "E"],
#               use_cache=USE_CACHE, save_to_cache=False, title=None)

## Actual Cases

In [ ]:
# Template
cases = []
subtitles = []
name = ""
title = None


# example_plots(cases, name, subtitles, title=title)
def print_names(cases, dID: str):
    named_cases = [f'{i}{dID}' for i in cases]
    print("Using cases:")
    for case in named_cases:
        print(generate_model_name(final_evaluation[case]))


def get_cases(cases, dID: str):
    return [f'{i}{dID}' for i in cases]

In [ ]:
# Best Cases for each model
cases = [1, 3, 9, 11, 17, 19]
subtitles = ["DiffTraj-uncond [1]", "DiffTraj-cond [3]", "UNetVAE-uncond [9]", "UNetVAE-cond [11]",
             "UNetGAN-uncond [17]", "UNetGAN-cond [19]"]
name = "examples_nodp_"
title = None

In [ ]:
# Porto
print_names(cases, 'p')

# Plot
example_plots(get_cases(cases, 'p'), name + 'porto', subtitles, title=title)

In [ ]:
# GeoLife
print_names(cases, 'g')

# Plot
example_plots(get_cases(cases, 'g'), name + 'geolife', subtitles, title=title)

In [ ]:
# Threat Model D, e_c = 10.0
cases = [2, 8, 10, 16, 18, 24]
subtitles = ['DP-DiffTraj-uncond [2]',
             'DP-DiffTraj-DP-cond (Sample) [8]',
             'DP-UNetVAE-uncond [10]',
             'DP-UNetVAE-DP-cond (Sample) [16]',
             'DP-UNetGAN-uncond [18]',
             'DP-UNetGAN-DP-cond (Sample) [24]']
name = "examples_dp_ec10_"
title = None

In [ ]:
# Porto
print_names(cases, 'p')

# Plot
example_plots(get_cases(cases, 'p'), name + 'porto', subtitles, title=title)

In [ ]:
# GeoLife
print_names(cases, 'g')

# Plot
example_plots(get_cases(cases, 'g'), name + 'geolife', subtitles, title=title)

In [ ]:
# Threat Model D, e_c = 20.0
cases = [2, '8x', 10, '16x', 18, '24x']
subtitles = ['DP-DiffTraj-uncond [2]',
             'DP-DiffTraj-DP-cond (Sample) [8a]',
             'DP-UNetVAE-uncond [10]',
             'DP-UNetVAE-DP-cond (Sample) [16a]',
             'DP-UNetGAN-uncond [18]',
             'DP-UNetGAN-DP-cond (Sample) [24a]']
name = "examples_dp_ec20_"
title = None

In [ ]:
# Porto
print_names(cases, 'p')

# Plot
example_plots(get_cases(cases, 'p'), name + 'porto', subtitles, title=title)

In [ ]:
# GeoLife
print_names(cases, 'g')

# Plot
example_plots(get_cases(cases, 'g'), name + 'geolife', subtitles, title=title)

In [ ]:
# Threat Model C: Privacy wrt to the conditional information (test set)
cases = [5, 7, 13, 15, 21, 23]
subtitles = ['DiffTraj-DP-cond (8-Stat) [5]', 'DiffTraj-DP-cond (Sample) [7]', 'UNetVAE-DP-cond (8-Stat) [13]',
             'UNetVAE-DP-cond (Sample) [15]', 'UNetGAN-DP-cond (8-Stat) [21]', 'UNetGAN-DP-cond (Sample) [23]']
name = "examples_dpcond_ec10_"
title = None

In [ ]:
# Porto
print_names(cases, 'p')

# Plot
example_plots(get_cases(cases, 'p'), name + 'porto', subtitles, title=title, include_dataset=False)

In [ ]:
# GeoLife
print_names(cases, 'g')

# Plot
example_plots(get_cases(cases, 'g'), name + 'geolife', subtitles, title=title, include_dataset=False)

In [ ]:
# Threat Model C: Privacy wrt to the conditional information (test set) [e_c = 20]
cases = ["5x", "7x", "13x", "15x", "21x", "23x"]
subtitles = ['DiffTraj-DP-cond (8-Stat) [5a]', 'DiffTraj-DP-cond (Sample) [7a]', 'UNetVAE-DP-cond (8-Stat) [13a]',
             'UNetVAE-DP-cond (Sample) [15a]', 'UNetGAN-DP-cond (8-Stat) [21a]', 'UNetGAN-DP-cond (Sample) [23a]']
name = "examples_dpcond_ec20_"
title = None

In [ ]:
# Porto
print_names(cases, 'p')

# Plot
example_plots(get_cases(cases, 'p'), name + 'porto', subtitles, title=title, include_dataset=False)

In [ ]:
# GeoLife
print_names(cases, 'g')

# Plot
example_plots(get_cases(cases, 'g'), name + 'geolife', subtitles, title=title, include_dataset=False)